In [ ]:
# --- Imports & path setup -------------------------------------------------
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from data_loader import DEVICE, PROJECT_ROOT, get_dataset_info
from moo_pymoo import run_pymoo_study
from analyze_study import analyze

print(f"Device       : {DEVICE}")
print(f"Project root : {PROJECT_ROOT}")
if str(DEVICE) == "cuda":
    import torch
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# --- Configuration --------------------------------------------------------
DATASET           = "fashion_mnist"
POP_SIZE          = 40
N_GEN             = 5
TRAIN_SUBSET_SIZE = None        # None = full training set
USE_AMP           = True        # fp16 mixed precision (CUDA only)
INFERENCE_WARMUP  = 10
INFERENCE_TIMED   = 100
NUM_WORKERS       = 4
SEED              = 42

COMPARE_PARETO    = None        # set to a botorch pareto_front.csv for joint-front overlay

info = get_dataset_info(DATASET)
total_trials = POP_SIZE * N_GEN
print(f"Dataset        : {DATASET}  (classes={info['num_classes']}, native res={info['default_resolution']})")
print(f"Total trials   : {total_trials} (pop={POP_SIZE} × gen={N_GEN})")
print(f"Train subset   : {'FULL DATASET' if TRAIN_SUBSET_SIZE is None else f'{TRAIN_SUBSET_SIZE:,} samples'}")
print(f"AMP            : {USE_AMP and str(DEVICE)=='cuda'}")

In [1]:
summary, study_dir = run_pymoo_study(
    dataset_name=DATASET,
    pop_size=POP_SIZE,
    n_gen=N_GEN,
    train_subset_size=TRAIN_SUBSET_SIZE,
    seed=SEED,
    num_workers=NUM_WORKERS,
    use_amp=USE_AMP,
    inference_warmup=INFERENCE_WARMUP,
    inference_timed=INFERENCE_TIMED,
)

print("\n" + "=" * 70)
print(f"Study dir : {study_dir}")
print(f"Wall time : {summary['elapsed_seconds']/60:.1f} min")
print(f"Pareto    : {summary['n_pareto_points']} non-dominated points (final population)")

  trial 000 | acc=0.6614 | ms=  0.137 | params=    4,970 |   7.6s | plain/1L/8ch/256fc/bs128
  trial 001 | acc=0.9084 | ms=  2.638 | params=  258,762 |  10.1s | plain/4L/32ch/128fc/bs256
  trial 002 | acc=0.8303 | ms=  0.905 | params=   23,818 |   7.9s | plain/2L/32ch/64fc/bs128
  trial 003 | acc=0.8273 | ms=  1.113 | params=   15,914 |  10.1s | plain/2L/16ch/256fc/bs256
  trial 004 | acc=0.8318 | ms=  1.054 | params=    4,762 |  12.4s | plain/2L/8ch/128fc/bs128
  trial 005 | acc=0.9135 | ms=  1.320 | params=  276,554 |  18.0s | plain/4L/32ch/256fc/bs64
  trial 006 | acc=0.6423 | ms=  0.165 | params=    1,930 |  13.1s | plain/1L/16ch/64fc/bs256
  trial 007 | acc=0.8912 | ms=  2.011 | params=  249,866 |  10.6s | plain/4L/32ch/64fc/bs256
  trial 008 | acc=0.5145 | ms=  0.161 | params=   11,402 |  19.9s | plain/1L/32ch/256fc/bs64
  trial 009 | acc=0.7465 | ms=  0.135 | params=   11,402 |   7.5s | plain/1L/32ch/256fc/bs256
  trial 010 | acc=0.8391 | ms=  0.477 | params=   28,618 |  16.0s |